In [1]:
# imports
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
# Make torch deterministic
_ = torch.manual_seed(0)

In [3]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

# Load the train dataset
mnist_trainset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# Load the test dataset
mnist_testset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(mnist_testset, batch_size=10, shuffle=True)

# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 487kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.05MB/s]


In [4]:
# Making a complex neural network (So that we can see the effect of lora clearly!)

class RichNeuralNet(nn.Module):
  def __init__(self, hidden_size_1=1000, hidden_size_2=2000):
    super().__init__()
    self.linear1 = nn.Linear(28*28, hidden_size_1)
    self.linear2 = nn.Linear(hidden_size_1, hidden_size_2)
    self.linear3 = nn.Linear(hidden_size_2, 10)
    self.relu = nn.ReLU()

  def forward(self, img):
    x = img.view(-1, 28*28)
    x = self.relu(self.linear1(x))
    x = self.relu(self.linear2(x))
    x = self.linear3(x)

    return x

net = RichNeuralNet().to(device)

In [6]:
# Now we will train the complex neural network on the MNIST Dataset.

def train(train_loader, net, epochs=5, total_iteration_limit=None):
  criterian = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

  total_iterations = 0

  for epoch in range(epochs):
    net.train()
    loss_sum = 0
    num_iterations = 0
    data_iterator = tqdm(train_loader, desc=f'Epoch: {epoch+1}/{epochs}')

    if total_iteration_limit is not None:
      data_iterator.total = total_iteration_limit

    for data in data_iterator:
      total_iterations += 1
      num_iterations += 1
      X, y = data
      X = X.to(device)
      y =y.to(device)
      optimizer.zero_grad()
      output = net(X.view(-1, 28*28))
      loss = criterian(output,  y)
      loss_sum += loss.item()
      avg_loss = loss_sum / num_iterations
      data_iterator.set_postfix(loss=avg_loss)
      loss.backward()
      optimizer.step()

      if total_iteration_limit is not None and total_iterations >= total_iteration_limit:
        return

train(train_loader, net, epochs=1)

Epoch: 1/1: 100%|██████████| 6000/6000 [07:25<00:00, 13.46it/s, loss=0.236]


In [7]:
original_weights = {}
for name, param in net.named_parameters():
  original_weights[name] = param.clone().detach()

In [9]:
def test():

  correct = 0
  total = 0
  wrong_counts = [0 for i in range(10)]

  with torch.no_grad():
    for data in tqdm(test_loader, desc='Testing'):
      X, y = data
      X = X.to(device)
      y = y.to(device)
      output = net(X.view(-1, 28*28))
      for idx, i in enumerate(output):
        if torch.argmax(i) == y[idx]:
          correct += 1
        else:
          wrong_counts[y[idx]] += 1
        total += 1
  print(f"Accuracy: {round(correct/total)}.")
  for i in range(len(wrong_counts)):
    print(f"Wrong counts for the digit {i}: {wrong_counts[i]}")

test()

Testing: 100%|██████████| 1000/1000 [00:05<00:00, 170.30it/s]

Accuracy: 1.
Wrong counts for the digit 0: 31
Wrong counts for the digit 1: 17
Wrong counts for the digit 2: 46
Wrong counts for the digit 3: 74
Wrong counts for the digit 4: 29
Wrong counts for the digit 5: 7
Wrong counts for the digit 6: 36
Wrong counts for the digit 7: 80
Wrong counts for the digit 8: 25
Wrong counts for the digit 9: 116


In [10]:
# Number of parameters
total_original_params = 0
for index, layer in enumerate([net.linear1, net.linear2, net.linear3]):
  total_original_params += layer.weight.nelement() + layer.bias.nelement()
  print(f"layer {index+1} - w: {layer.weight.shape}, B: {layer.bias.shape}")
print(f"Total number of parameters: {total_original_params}.")

layer 1 - w: torch.Size([1000, 784]), B: torch.Size([1000])
layer 2 - w: torch.Size([2000, 1000]), B: torch.Size([2000])
layer 3 - w: torch.Size([10, 2000]), B: torch.Size([10])
Total number of parameters: 2807010.


In [18]:
class LoRAParametrization(nn.Module):
  def __init__(self, feature_in, feature_out, rank=1, alpha=1, device='cpu'):
    super().__init__()
    self.lora_A = nn.Parameter(torch.zeros((rank, feature_out)).to(device))
    self.lora_B = nn.Parameter(torch.zeros((feature_in, rank)).to(device))
    nn.init.normal_(self.lora_A, mean=0, std=1)

    self.scale = alpha/rank
    self.enabled = True

  def forward(self, original_weights):
    if self.enabled:
      # Return W + (B*A)*Scale
      return original_weights + torch.matmul(self.lora_B, self.lora_A).view(original_weights.shape) * self.scale
    else:
      return original_weights

In [27]:
import torch.nn.utils.parametrize as parametrize


def linear_layer_parameterization(layer, device, rank=1, lora_alpha=1):
  # We will only add parameterization to the weight matrix and not to bias matrix
  features_in, features_out = layer.weight.shape
  return LoRAParametrization(
      features_in,
      features_out,
      rank=rank,
      alpha=lora_alpha,
      device=device,
  )

parametrize.register_parametrization(
    net.linear1, 'weight', linear_layer_parameterization(net.linear1, device)
)

parametrize.register_parametrization(
    net.linear2, 'weight', linear_layer_parameterization(net.linear2, device)
)

parametrize.register_parametrization(
    net.linear3, 'weight', linear_layer_parameterization(net.linear3, device)
)

def enable_disable_lora(enabled=True):
  for layer in [net.linear1, net.linear2, net.linear3]:
    layer.parametrizations["weight"][0].enabled = enabled

In [22]:
total_parameters_lora = 0
total_parameters_non_lora = 0
for index, layer in enumerate([net.linear1, net.linear2, net.linear3]):
  total_parameters_lora += layer.parametrizations["weight"][0].lora_A.nelement() + layer.parametrizations["weight"][0].lora_B.nelement()
  total_parameters_non_lora += layer.weight.nelement() + layer.bias.nelement()
  print(
      f'layer: {index+1} | W: {layer.weight.shape} | B: {layer.bias.shape} | Lora_A: {layer.parametrizations["weight"][0].lora_A.shape} | Lora_B: {layer.parametrizations["weight"][0].lora_B.shape}'
  )

assert total_parameters_non_lora == total_original_params
print(f"Total number of parameters (original): {total_parameters_non_lora}")
print(f"Total number of parameters (original + LoRA): {total_parameters_lora + total_parameters_non_lora}")
print(f"Parameters introduced by LoRA: {total_parameters_lora}")
print(f"Parameter increment: {(total_parameters_lora/total_parameters_non_lora * 100):.3f}%")

layer: 1 | W: torch.Size([1000, 784]) | B: torch.Size([1000]) | Lora_A: torch.Size([1, 784]) | Lora_B: torch.Size([1000, 1])
layer: 2 | W: torch.Size([2000, 1000]) | B: torch.Size([2000]) | Lora_A: torch.Size([1, 1000]) | Lora_B: torch.Size([2000, 1])
layer: 3 | W: torch.Size([10, 2000]) | B: torch.Size([10]) | Lora_A: torch.Size([1, 2000]) | Lora_B: torch.Size([10, 1])
Total number of parameters (original): 2807010
Total number of parameters (original + LoRA): 2813804
Parameters introduced by LoRA: 6794
Parameter increment: 0.242%


In [23]:
# Freeze the non-lora parameters
for name, param in net.named_parameters():
  if 'lora' not in name:
    print(f"Freezing non-lora parameters {name}.")
    param.requires_grad = False

Freezing non-lora parameters linear1.bias.
Freezing non-lora parameters linear1.parametrizations.weight.original.
Freezing non-lora parameters linear2.bias.
Freezing non-lora parameters linear2.parametrizations.weight.original.
Freezing non-lora parameters linear3.bias.
Freezing non-lora parameters linear3.parametrizations.weight.original.


In [24]:
# Load the MNIST dataset again, by keeping only the digit 9
mnist_trainset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
include_indices = mnist_trainset.targets == 9
mnist_trainset.data = mnist_trainset.data[include_indices]
mnist_trainset.targets = mnist_trainset.targets[include_indices]
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

train(train_loader, net, epochs=1)

Epoch: 1/1: 100%|██████████| 595/595 [00:20<00:00, 29.09it/s, loss=0.0256]


In [28]:
# Check all the frozen parameters are still unchanged after finetuning
assert torch.all(net.linear1.parametrizations.weight.original == original_weights['linear1.weight'])
assert torch.all(net.linear2.parametrizations.weight.original == original_weights['linear2.weight'])
assert torch.all(net.linear3.parametrizations.weight.original == original_weights['linear3.weight'])

# If we enable lora, the the net.linear.weight will have the new weights original + LoRA and net.linear1.parametrizations.weight.original will give the original weights.
enable_disable_lora(enabled=True)
assert torch.equal(net.linear1.weight, net.linear1.parametrizations.weight.original + (net.linear1.parametrizations.weight[0].lora_B @ net.linear1.parametrizations.weight[0].lora_A) * net.linear1.parametrizations.weight[0].scale)

# If we disable lora, the net.linear1.weight is the original one
enable_disable_lora(enabled=False)
assert torch.equal(net.linear1.weight, original_weights['linear1.weight'])

In [29]:
enable_disable_lora(enabled=True)
test()

Testing: 100%|██████████| 1000/1000 [00:23<00:00, 43.13it/s]

Accuracy: 0.
Wrong counts for the digit 0: 412
Wrong counts for the digit 1: 438
Wrong counts for the digit 2: 633
Wrong counts for the digit 3: 830
Wrong counts for the digit 4: 671
Wrong counts for the digit 5: 467
Wrong counts for the digit 6: 68
Wrong counts for the digit 7: 871
Wrong counts for the digit 8: 727
Wrong counts for the digit 9: 0


In [30]:
enable_disable_lora(enabled=False)
test()

Testing: 100%|██████████| 1000/1000 [00:14<00:00, 68.81it/s]

Accuracy: 1.
Wrong counts for the digit 0: 31
Wrong counts for the digit 1: 17
Wrong counts for the digit 2: 46
Wrong counts for the digit 3: 74
Wrong counts for the digit 4: 29
Wrong counts for the digit 5: 7
Wrong counts for the digit 6: 36
Wrong counts for the digit 7: 80
Wrong counts for the digit 8: 25
Wrong counts for the digit 9: 116
